# 06 因子构建：把价格变成可检验的特征

## 6.1 本章要解决什么问题

因子是把原始行情、成交量、财务数据或其他信息加工成的可计算特征。它不是交易结论，而是后续排序、检验、组合构建会消费的一张特征表。

本章使用本地缓存的真实 ETF 日线价格，完成三件事：

1. 在单只 ETF 上手写 `momentum_60`、`low_vol_20`、`ma_gap_20_60` 的公式。
2. 把同样的公式扩展成多 ETF 的 `date, code, factor...` 因子面板。
3. 用 `lib.factors` 生成实战版面板，并说明 `shift(1)` 如何避免未来函数。

## 6.2 输入与输出

- 输入：`data/sample/prices.parquet`，第 04 章缓存的 AKShare ETF OHLCV 数据。
- 上游：第 05 章已经讲过交易日对齐和收益率矩阵。
- 输出：`outputs/results/chapter06_factor_panel.csv`，供人工检查；第 07 章会用同样的因子面板做 IC 与分层检验。


In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "lib").exists() and (candidate / "data").exists():
            return candidate
        nested = candidate / "pyquant-roadmap"
        if (nested / "lib").exists() and (nested / "data").exists():
            return nested
    raise RuntimeError("Cannot find pyquant-roadmap project root")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

from lib.data import load_sample_prices
from lib.paths import RESULTS_DIR

prices = load_sample_prices().copy()
prices["date"] = pd.to_datetime(prices["date"])
prices["code"] = prices["code"].astype(str)
prices = prices.sort_values(["date", "code"]).reset_index(drop=True)

close = prices.pivot(index="date", columns="code", values="close").sort_index().astype(float)
returns = close.pct_change()

price_info = pd.Series(
    {
        "price_rows": len(prices),
        "close_shape": close.shape,
        "start_date": close.index.min().date(),
        "end_date": close.index.max().date(),
        "codes": ", ".join(close.columns.astype(str)),
    }
)

display(price_info.to_frame("value"))
display(close.tail())


,value
price_rows,2900
close_shape,"(725, 4)"
start_date,2021-01-04
end_date,2023-12-29
codes,"159915, 510300, 510500, 512100"


code,159915,510300,510500,512100
date,,,,
2023-12-25,1.783,3.135,5.163,2.232
2023-12-26,1.761,3.115,5.116,2.202
2023-12-27,1.763,3.125,5.132,2.215
2023-12-28,1.833,3.209,5.235,2.262
2023-12-29,1.842,3.219,5.279,2.296


## 6.3 因子就是可计算特征

本章先构造三个技术因子。它们都只依赖历史收盘价和收益率，便于看清楚公式和日期对齐。

- `momentum_60`：过去 60 个交易日的涨跌幅。越高表示中期走势越强。
- `low_vol_20`：过去 20 个交易日收益率标准差的相反数。越高表示波动越低。
- `ma_gap_20_60`：20 日均线相对 60 日均线的偏离。越高表示短期价格中枢更强。

后续组合不直接消费价格矩阵，而是消费因子面板。面板的每一行表示“某个日期、某个标的、这些因子的取值”。


In [2]:
selected_code = "510300" if "510300" in close.columns else str(close.columns[0])
one_etf = pd.DataFrame(
    {
        "close": close[selected_code],
        "daily_return": returns[selected_code],
    }
)

print(f"Selected ETF for handwritten formulas: {selected_code}")
display(one_etf.tail(8))


Selected ETF for handwritten formulas: 510300


,close,daily_return
date,,
2023-12-20,3.089,-0.009936
2023-12-21,3.120,0.010036
2023-12-22,3.126,0.001923
2023-12-25,3.135,0.002879
2023-12-26,3.115,-0.006380
2023-12-27,3.125,0.003210
2023-12-28,3.209,0.026880
2023-12-29,3.219,0.003116


## 6.4 手写最小实现：一只 ETF 的三个公式

先不要急着封装。对一只 ETF 来说，因子就是一列可以从历史价格推出来的数。

这里约定：因子日期 `t` 表示“在 `t` 日开盘前或调仓前已经知道的信息”。因此公式统一使用 `shift(1)`，只让 `t-1` 及更早的数据参与计算。

- `momentum_60(t) = close(t-1) / close(t-61) - 1`
- `low_vol_20(t) = -std(return(t-20), ..., return(t-1))`
- `ma_gap_20_60(t) = MA20(t-1) / MA60(t-1) - 1`


In [3]:
def momentum_60_one(close_s: pd.Series) -> pd.Series:
    return close_s.shift(1) / close_s.shift(61) - 1.0


def low_vol_20_one(close_s: pd.Series) -> pd.Series:
    ret = close_s.pct_change()
    return -ret.rolling(20).std().shift(1)


def ma_gap_20_60_one(close_s: pd.Series) -> pd.Series:
    ma20 = close_s.rolling(20).mean()
    ma60 = close_s.rolling(60).mean()
    return (ma20 / ma60 - 1.0).shift(1)


factor_cols = ["momentum_60", "low_vol_20", "ma_gap_20_60"]
one_factors = pd.DataFrame(
    {
        "close": close[selected_code],
        "momentum_60": momentum_60_one(close[selected_code]),
        "low_vol_20": low_vol_20_one(close[selected_code]),
        "ma_gap_20_60": ma_gap_20_60_one(close[selected_code]),
    }
)

display(one_factors.dropna().head(6))


,close,momentum_60,low_vol_20,ma_gap_20_60
date,,,,
2021-04-08,4.690,-0.033863,-0.014252,-0.055563
2021-04-09,4.619,-0.050992,-0.014234,-0.053646
2021-04-12,4.535,-0.074534,-0.013183,-0.053378
2021-04-13,4.528,-0.112177,-0.013699,-0.053451
2021-04-14,4.556,-0.108661,-0.012895,-0.052621
2021-04-15,4.528,-0.091888,-0.012865,-0.052133


## 6.5 为什么要 `shift(1)`

如果因子日期是 `t`，但公式用了 `close(t)` 才能算出来，那么这个信号在 `t` 日调仓前并不可得。研究时它可能看起来更好，但实盘无法复制，这就是常见的未来函数。

下面取第一条有效因子，手工拆开它实际使用的日期。重点看：因子日期比最后使用的收盘价晚一个交易日。


In [4]:
example_date = one_factors.dropna().index[0]
pos = close.index.get_loc(example_date)
last_known_date = close.index[pos - 1]
momentum_start_date = close.index[pos - 61]
ret_window = returns[selected_code].iloc[pos - 20 : pos]
ma_window = close[selected_code].iloc[:pos]

trace = pd.Series(
    {
        "factor_date": example_date.date(),
        "last_close_used": last_known_date.date(),
        "momentum_start_date": momentum_start_date.date(),
        "return_window_start": ret_window.index.min().date(),
        "return_window_end": ret_window.index.max().date(),
    }
)

hand_check = pd.Series(
    {
        "momentum_60": close.loc[last_known_date, selected_code] / close.loc[momentum_start_date, selected_code] - 1.0,
        "low_vol_20": -ret_window.std(),
        "ma_gap_20_60": ma_window.tail(20).mean() / ma_window.tail(60).mean() - 1.0,
    }
)

check_values = pd.DataFrame(
    {
        "hand_check": hand_check,
        "factor_table": one_factors.loc[example_date, factor_cols],
    }
)

display(trace.to_frame("value"))
display(check_values)


,value
factor_date,2021-04-08
last_close_used,2021-04-07
momentum_start_date,2021-01-04
return_window_start,2021-03-10
return_window_end,2021-04-07


,hand_check,factor_table
momentum_60,-0.033863,-0.033863
low_vol_20,-0.014252,-0.014252
ma_gap_20_60,-0.055563,-0.055563


## 6.6 从一只 ETF 扩展到多 ETF

实际策略需要在同一天比较多只 ETF。pandas 的矩阵计算适合先生成“日期 x 标的”的宽表，再把它整理成长表。

长表面板的好处是：

- 第 07 章可以按 `date` 做横截面 IC。
- 第 08 章可以按 `date` 排序选 TopN。
- 之后的回测和交易信号可以沿用同一套 `date, code` 主键。


In [5]:
def build_factor_matrices(close_df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    ret = close_df.pct_change()
    ma20 = close_df.rolling(20).mean()
    ma60 = close_df.rolling(60).mean()
    return {
        "momentum_60": close_df.shift(1) / close_df.shift(61) - 1.0,
        "low_vol_20": -ret.rolling(20).std().shift(1),
        "ma_gap_20_60": (ma20 / ma60 - 1.0).shift(1),
    }


manual_matrices = build_factor_matrices(close)
manual_wide = pd.concat(manual_matrices, axis=1)
manual_panel = (
    manual_wide.stack(level=1, future_stack=True)
    .rename_axis(index=["date", "code"])
    .reset_index()
    .dropna(subset=factor_cols)
    .sort_values(["date", "code"])
    .reset_index(drop=True)
)
manual_panel = manual_panel[["date", "code", *factor_cols]]

print(f"manual_panel shape: {manual_panel.shape}")
print(f"factor dates: {manual_panel['date'].min().date()} -> {manual_panel['date'].max().date()}")
print(f"codes: {', '.join(sorted(manual_panel['code'].unique()))}")
display(manual_panel.head(8))
display(manual_panel.tail(8))


manual_panel shape: (2656, 5)
factor dates: 2021-04-08 -> 2023-12-29
codes: 159915, 510300, 510500, 512100


,date,code,momentum_60,low_vol_20,ma_gap_20_60
0,2021-04-08,159915,-0.090390,-0.017595,-0.088141
1,2021-04-08,510300,-0.033863,-0.014252,-0.055563
2,2021-04-08,510500,-0.019122,-0.009620,-0.025919
3,2021-04-08,512100,-0.048482,-0.011525,-0.019949
4,2021-04-09,159915,-0.087015,-0.017401,-0.084311
5,2021-04-09,510300,-0.050992,-0.014234,-0.053646
6,2021-04-09,510500,-0.024881,-0.009134,-0.023002
7,2021-04-09,512100,-0.052694,-0.011090,-0.016187


,date,code,momentum_60,low_vol_20,ma_gap_20_60
2648,2023-12-28,159915,-0.090299,-0.010501,-0.035640
2649,2023-12-28,510300,-0.104071,-0.007588,-0.042040
2650,2023-12-28,510500,-0.070121,-0.006853,-0.015941
2651,2023-12-28,512100,-0.052609,-0.008806,-0.011384
2652,2023-12-29,159915,-0.062883,-0.014133,-0.035679
2653,2023-12-29,510300,-0.082619,-0.010009,-0.041748
2654,2023-12-29,510500,-0.055736,-0.008525,-0.016505
2655,2023-12-29,512100,-0.038674,-0.010368,-0.013011


## 6.7 读懂一张横截面

下面取最后一个因子日期。每一行都是同一日期下某只 ETF 的三个特征。此时还没有判断因子是否有效，只是在准备第 07 章要检验的输入。


In [6]:
latest_date = manual_panel["date"].max()
latest_cross_section = manual_panel[manual_panel["date"].eq(latest_date)].sort_values(
    "momentum_60", ascending=False
)

display(latest_cross_section.round(4))


,date,code,momentum_60,low_vol_20,ma_gap_20_60
2655,2023-12-29,512100,-0.0387,-0.0104,-0.0130
2654,2023-12-29,510500,-0.0557,-0.0085,-0.0165
2652,2023-12-29,159915,-0.0629,-0.0141,-0.0357
2653,2023-12-29,510300,-0.0826,-0.0100,-0.0417


## 6.8 用 `lib.factors` 生成实战版因子面板

手写公式用于理解原理；可复用流程应该放进 `lib/`。`build_technical_factor_panel` 保留同样的字段名，内部用 pandas 和 `ta` 的均线工具生成面板。

这里把手写面板和 `lib` 面板做一次差异检查。差异应接近 0，说明封装没有改变公式含义。


In [7]:
from lib.factors import build_technical_factor_panel, zscore_by_date

lib_panel = build_technical_factor_panel(prices)

print(f"lib_panel shape: {lib_panel.shape}")
print(f"factor dates: {lib_panel['date'].min().date()} -> {lib_panel['date'].max().date()}")
display(lib_panel.head(8))

comparison = manual_panel.merge(lib_panel, on=["date", "code"], suffixes=("_manual", "_lib"))
max_abs_diff = pd.Series(
    {
        col: (comparison[f"{col}_manual"] - comparison[f"{col}_lib"]).abs().max()
        for col in factor_cols
    }
)

display(max_abs_diff.to_frame("max_abs_diff"))
assert (max_abs_diff < 1e-12).all()


lib_panel shape: (2656, 5)
factor dates: 2021-04-08 -> 2023-12-29


,date,code,momentum_60,low_vol_20,ma_gap_20_60
0,2021-04-08,159915,-0.090390,-0.017595,-0.088141
1,2021-04-08,510300,-0.033863,-0.014252,-0.055563
2,2021-04-08,510500,-0.019122,-0.009620,-0.025919
3,2021-04-08,512100,-0.048482,-0.011525,-0.019949
4,2021-04-09,159915,-0.087015,-0.017401,-0.084311
5,2021-04-09,510300,-0.050992,-0.014234,-0.053646
6,2021-04-09,510500,-0.024881,-0.009134,-0.023002
7,2021-04-09,512100,-0.052694,-0.011090,-0.016187


,max_abs_diff
momentum_60,0.0
low_vol_20,0.0
ma_gap_20_60,0.0


## 6.9 横截面标准化：为因子检验做准备

不同因子的量纲不同，不能直接相加或比较大小。常见做法是在每个日期的横截面上做 z-score：减去当日均值，再除以当日标准差。

第 07 章会用标准化后的因子和未来收益做 IC 检验。本章先看一眼最后一个日期的标准化结果。


In [8]:
z_panel = zscore_by_date(lib_panel, factor_cols)
z_cols = [f"{col}_z" for col in factor_cols]
latest_z = z_panel[z_panel["date"].eq(z_panel["date"].max())].sort_values("momentum_60_z", ascending=False)

display(latest_z[["date", "code", *factor_cols, *z_cols]].round(4))


,date,code,momentum_60,low_vol_20,ma_gap_20_60,momentum_60_z,low_vol_20_z,ma_gap_20_60_z
2655,2023-12-29,512100,-0.0387,-0.0104,-0.0130,1.3522,0.1888,1.1221
2654,2023-12-29,510500,-0.0557,-0.0085,-0.0165,0.2693,1.0808,0.8365
2652,2023-12-29,159915,-0.0629,-0.0141,-0.0357,-0.1844,-1.6324,-0.7312
2653,2023-12-29,510300,-0.0826,-0.0100,-0.0417,-1.4370,0.3628,-1.2274


## 6.10 保存本章产出

保存的是未标准化因子面板。原因是原始因子更适合排查公式、窗口和缺失值；标准化可以在第 07 章根据研究任务重新计算。


In [9]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
factor_panel_path = RESULTS_DIR / "chapter06_factor_panel.csv"
lib_panel.to_csv(factor_panel_path, index=False, encoding="utf-8-sig")

output_summary = pd.Series(
    {
        "path": str(factor_panel_path.relative_to(PROJECT_ROOT)),
        "rows": len(lib_panel),
        "columns": ", ".join(lib_panel.columns),
        "start_date": lib_panel["date"].min().date(),
        "end_date": lib_panel["date"].max().date(),
    }
)

display(output_summary.to_frame("value"))


,value
path,outputs\results\chapter06_factor_panel.csv
rows,2656
columns,"date, code, momentum_60, low_vol_20, ma_gap_20_60"
start_date,2021-04-08
end_date,2023-12-29


## 6.11 质量检查与常见坑

构造因子时至少检查三件事：

- 主键是否唯一：同一个 `date, code` 不应该重复。
- 因子值是否仍有缺失：窗口不足的早期日期应该已经被剔除。
- 日期范围是否符合预期：60 日窗口加 `shift(1)` 后，第一条有效日期会晚于原始价格开始日期。

最常见的坑是日期约定混乱：有的因子用了 `shift(1)`，有的没有。只要后续要把因子和未来收益对齐，就必须先统一“因子日期代表什么”。


In [10]:
quality_checks = pd.Series(
    {
        "duplicate_date_code": int(lib_panel.duplicated(["date", "code"]).sum()),
        "missing_factor_values": int(lib_panel[factor_cols].isna().sum().sum()),
        "unique_dates": lib_panel["date"].nunique(),
        "unique_codes": lib_panel["code"].nunique(),
        "first_price_date": close.index.min().date(),
        "first_factor_date": lib_panel["date"].min().date(),
    }
)

missing_by_factor = lib_panel[factor_cols].isna().sum().rename("missing_count")

display(quality_checks.to_frame("value"))
display(missing_by_factor.to_frame())

assert quality_checks["duplicate_date_code"] == 0
assert quality_checks["missing_factor_values"] == 0


,value
duplicate_date_code,0
missing_factor_values,0
unique_dates,664
unique_codes,4
first_price_date,2021-01-04
first_factor_date,2021-04-08


,missing_count
momentum_60,0
low_vol_20,0
ma_gap_20_60,0


## 6.12 练习：把低波动窗口改成 60 日

把 `low_vol_20` 改成 `low_vol_60` 只需要替换滚动窗口，同时保留 `shift(1)`。观察它的第一条有效日期会比 `low_vol_20` 更晚。


In [11]:
low_vol_60 = -returns.rolling(60).std().shift(1)
low_vol_60_panel = (
    low_vol_60.stack(future_stack=True)
    .rename("low_vol_60")
    .reset_index()
    .rename(columns={"level_0": "date", "level_1": "code"})
    .dropna()
    .sort_values(["date", "code"])
    .reset_index(drop=True)
)

exercise_summary = pd.Series(
    {
        "low_vol_60_rows": len(low_vol_60_panel),
        "first_low_vol_60_date": low_vol_60_panel["date"].min().date(),
        "chapter_factor_first_date": lib_panel["date"].min().date(),
    }
)

display(exercise_summary.to_frame("value"))
display(low_vol_60_panel.tail(8).round(4))


,value
low_vol_60_rows,2656
first_low_vol_60_date,2021-04-08
chapter_factor_first_date,2021-04-08


,date,code,low_vol_60
2648,2023-12-28,159915,-0.0116
2649,2023-12-28,510300,-0.0077
2650,2023-12-28,510500,-0.0084
2651,2023-12-28,512100,-0.0101
2652,2023-12-29,159915,-0.0127
2653,2023-12-29,510300,-0.0085
2654,2023-12-29,510500,-0.0088
2655,2023-12-29,512100,-0.0104


## 6.13 交给第 07 章

到这里，因子构建的输出已经明确：一张以 `date, code` 为主键的多因子面板。

第 07 章要回答的问题不是“因子怎么算”，而是“因子排序和未来收益有没有关系”。它会读取同一批缓存价格，重新生成或读取本章这种结构的因子面板，然后：

- 对每个日期做横截面标准化。
- 把因子日期和未来 5 日收益对齐。
- 计算 IC 序列和分层收益，判断这些特征是否值得进入组合构建。
